## Executive Summary

**Purpose:** Explain what a single-stage planetary gear train is made of and what every gear parameter means, with labelled drawings, before the calculation notebooks use them.

**What it covers:**

1. The parts of the stage and their jobs (front view and axial section).
2. How torque and speed pass through it, and the three ways it can be arranged.
3. Spur gear tooth nomenclature on an external and an internal gear: module, pitch, pressure angle, reference, base, tip and root diameters, addendum, dedendum, tooth thickness, space width, face width, profile shift.
4. Two gears in mesh: centre distance, line of action, operating pressure angle, path of contact, contact ratio, backlash.
5. The list of data that defines a design, from requirements to the manufacturing package.
6. A glossary of every parameter with its symbol, units, and the notebook that calculates it.

The drawings are made from the library's own geometry (`emachines.gears`), so the teeth are true involutes at the stated sizes.

## How It Fits Into the Gear Design Workflow

This page has no calculations of its own. The workflow it introduces:

```
00_overview  →  01_involute  →  02_layout  →  03_rating  →  04_components  →  05_manufacturing  →  06_design
 (this page)    tooth geometry   tooth counts   ISO 6336      bearings, pins,    tolerances, data     search + files
                and inspection   and module     strength      carrier, key       sheets, DXF
```

In [ ]:
#| hide
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle, Arc, FancyArrowPatch, Polygon, Wedge
from emachines.gears.involute import GearGeometry, inv, mesh_check
from emachines.gears.layout import PlanetarySet
from emachines.gears.rating import RatingInputs, rate_planetary
from emachines.gears.components import ComponentInputs, size_components, rim_thicknesses
from emachines.gears.manufacturing import gear_outline

INK, BLUE, RED, GREEN = "#222222", "#1f5fa8", "#b8322a", "#2a8a4a"
C_SUN, C_PLANET, C_RING, C_CARRIER, C_STEEL = "#d8e8d0", "#f6e7c8", "#dfe7f1", "#cfe8d6", "#bdbdbd"


def rot(P, a):
    "Rotate points P (N×2) by angle a [rad] counter-clockwise."
    c, s = np.cos(a), np.sin(a)
    return P @ np.array([[c, s], [-s, c]])


def label(ax, text, xy, xytext, color=INK, ha="left", fs=9):
    "Leader-line label."
    ax.annotate(text, xy=xy, xytext=xytext, fontsize=fs, color=color, ha=ha, va="center",
                arrowprops=dict(arrowstyle="-", color=color, lw=0.7, shrinkA=0, shrinkB=0))


def dim(ax, p0, p1, text, off=(0, 0), color=BLUE, fs=9, both=True):
    "Dimension arrow between p0 and p1 with text at the midpoint (+off)."
    ax.annotate("", xy=p1, xytext=p0, arrowprops=dict(arrowstyle="<->" if both else "->", color=color, lw=0.8,
                                                       shrinkA=0, shrinkB=0))
    mx, my = (p0[0] + p1[0]) / 2 + off[0], (p0[1] + p1[1]) / 2 + off[1]
    ax.text(mx, my, text, color=color, fontsize=fs, ha="center", va="center",
            bbox=dict(fc="white", ec="none", pad=0.6, alpha=0.85))


def circ(ax, r, style="--", color=BLUE, lw=0.7, c=(0, 0)):
    ax.add_patch(Circle(c, r, fill=False, ls=style, ec=color, lw=lw))


# the example stage used for the drawings: 18/27/72, module 2, three planets, 20 mm face
ps = PlanetarySet.build(18, 27, 72, m=2.0, n_planets=3, b=20)
ci = ComponentInputs(envelope_d=170, shaft_d=16, envelope_length=50)
rat = rate_planetary(ps, RatingInputs(torque_out=150.0, n_in=1500.0), rims=rim_thicknesses(ps, ci))
comp = size_components(ps, rat, ci, life_h=10_000, n_in=1500.0)

## 1. The Parts of a Single-Stage Planetary

A planetary (epicyclic) stage has three coaxial members, the **sun gear**, the **ring gear** and the **planet carrier**, and a set of **planet gears** that mesh with the sun on the inside and with the ring on the outside. Each planet spins on a **pin** fixed in the carrier. Holding one of the three members still and driving another turns the third.

In the arrangement used throughout this module, the ring is held in the housing, the sun is driven by the input shaft (the motor), and the carrier is the output. The planets roll around inside the ring, and they carry the carrier with them at a lower speed and higher torque.

| Part | What it does | Sized in |
|---|---|---|
| Sun gear | Input; small external gear on the input shaft, meshes with every planet | `01_involute`, `02_layout`, `03_rating` |
| Planet gears ($N$ off) | Carry the load from the sun to the ring and to their pins; each one is an idler, loaded on both flanks | `02_layout`, `03_rating` |
| Ring gear (annulus) | Internal gear, held in the housing; takes the reaction torque | `02_layout`, `03_rating`, `04_components` |
| Planet carrier | Output; two plates joined by posts that pass between the planets, holding the planet pins | `04_components` |
| Planet pins | Shafts the planets spin on, pressed into both carrier plates | `04_components` |
| Planet bearings | Needle roller cage assemblies between pin and planet bore | `04_components` |
| Thrust washers | Locate each planet axially between the carrier plates | `04_components` |
| Input shaft and key | Drive the sun; the key groove sets how thin the sun rim may get | `04_components` |
| Ring fixing | Bolts (or press fit and dowels) that stop the ring from turning | `04_components` |

In [ ]:
#| hide
fig, ax = plt.subplots(figsize=(12, 8))
g = ps
R_out = ci.envelope_d / 2
ax.add_patch(Circle((0, 0), R_out, fc=C_RING, ec=INK, lw=1.0))
rho = np.pi * g.planet.z / g.ring.z
ax.add_patch(Polygon(rot(gear_outline(g.ring), rho), closed=True, fc="white", ec=INK, lw=0.8))
# carrier plate outline (behind the gears) and posts between the planets
ax.add_patch(Circle((0, 0), comp.carrier["plate_d"] / 2, fill=False, ec=GREEN, ls="--", lw=1.0))
for k in range(g.n_planets):
    th = 2 * np.pi * (k + 0.5) / g.n_planets
    w, r0, r1 = comp.carrier["post_width"], g.sun.da / 2 + 3, comp.carrier["plate_d"] / 2
    P = np.array([[r0, -w / 2], [r1, -w / 2], [r1, w / 2], [r0, w / 2]])
    ax.add_patch(Polygon(rot(P, th), closed=True, fc=C_CARRIER, ec=GREEN, lw=0.8, hatch="////", alpha=0.9, zorder=0.5))
# planets with bearing and pin
for k in range(g.n_planets):
    th = 2 * np.pi * k / g.n_planets
    psi = th * (1 + g.sun.z / g.planet.z) + np.pi - np.pi / g.planet.z
    c = g.a_w * np.array([np.cos(th), np.sin(th)])
    ax.add_patch(Polygon(rot(gear_outline(g.planet), psi) + c, closed=True, fc=C_PLANET, ec=INK, lw=0.8))
    ax.add_patch(Circle(c, comp.bearing["Ew"] / 2, fc="white", ec=INK, lw=0.6))
    for j in range(comp.bearing["rollers"]):
        a = 2 * np.pi * j / comp.bearing["rollers"]
        rr = (comp.bearing["Fw"] + comp.bearing["Dw"]) / 2
        ax.add_patch(Circle(c + rr * np.array([np.cos(a), np.sin(a)]), comp.bearing["Dw"] / 2 * 0.9, fc="#999", ec=INK, lw=0.3))
    ax.add_patch(Circle(c, comp.pin["d"] / 2, fc=C_STEEL, ec=INK, lw=0.6))
# sun, shaft and key
ax.add_patch(Polygon(gear_outline(g.sun), closed=True, fc=C_SUN, ec=INK, lw=0.8))
ax.add_patch(Circle((0, 0), ci.shaft_d / 2, fc=C_STEEL, ec=INK, lw=0.6))
kb, kh, t1 = comp.sun_hub["key_b"], comp.sun_hub["key_h"], comp.sun_hub["t1"]
ax.add_patch(Rectangle((-kb / 2, ci.shaft_d / 2 - t1), kb, kh, fc="#888", ec=INK, lw=0.5))
# ring bolts
if comp.ring_mount["bolts"]:
    for j in range(comp.ring_mount["bolts"]):
        a = 2 * np.pi * (j + 0.25) / comp.ring_mount["bolts"]
        ax.add_patch(Circle(comp.ring_mount["bolt_circle_d"] / 2 * np.array([np.cos(a), np.sin(a)]),
                            comp.ring_mount["clearance_hole"] / 2, fc="white", ec=INK, lw=0.6))
# centre lines
for a in (0, np.pi / 2):
    ax.plot([-(R_out + 6) * np.cos(a), (R_out + 6) * np.cos(a)], [-(R_out + 6) * np.sin(a), (R_out + 6) * np.sin(a)],
            color=RED, lw=0.4, ls="-.")
circ(ax, g.a_w, "-.", RED, 0.5)
# rotation arrows
def spin(c, r, a0, a1, color):
    arc = FancyArrowPatch(c + r * np.array([np.cos(a0), np.sin(a0)]), c + r * np.array([np.cos(a1), np.sin(a1)]),
                          connectionstyle=f"arc3,rad={0.35 if a1 > a0 else -0.35}", arrowstyle="-|>", mutation_scale=12, color=color, lw=1.4)
    ax.add_patch(arc)
spin(np.zeros(2), 5.5, -0.4, 1.6, RED)                                     # sun: counter-clockwise
c0 = g.a_w * np.array([1, 0])
spin(c0, 15, 1.9, 0.6, RED)                                                 # planet: clockwise on its pin
spin(np.zeros(2), comp.carrier["plate_d"] / 2 + 3, -1.25, -0.55, GREEN)     # carrier: same way as the sun
ax.text(c0[0], c0[1] - 15, "planet spins\nopposite to sun", color=RED, fontsize=8, ha="center", va="center",
        bbox=dict(fc="white", ec="none", pad=0.4, alpha=0.8))
# labels, ordered top to bottom on each side so the leaders do not cross
L = R_out + 12
th1 = 2 * np.pi / 3
cp = g.a_w * np.array([np.cos(th1), np.sin(th1)])
left = [("Planet gear (×3)", cp + [-14, 14], INK),
        ("Needle roller bearing", cp + [-(comp.bearing["Ew"] / 2 - 1), 0], INK),
        ("Planet pin", cp + [-1, -1], INK),
        ("Parallel key (DIN 6885)", (-1.5, ci.shaft_d / 2 + 1), INK),
        ("Sun gear (input, $n_s$)", (-g.sun.d / 2 + 2, 3), INK),
        ("Carrier post (between planets)", (-(g.sun.da / 2 + 14), -2), GREEN),
        ("Input shaft", (-3, -4), INK)]
for (txt, xy, col), y in zip(left, (92, 76, 62, 44, 22, -18, -60)):
    label(ax, txt, xy, (-L, y), color=col, ha="right")
right = [("Ring gear (fixed in housing)", (R_out * 0.55, R_out * 0.79), INK),
         ("Ring teeth (internal)", (g.ring.da / 2 * np.cos(0.95), g.ring.da / 2 * np.sin(0.95)), INK),
         ("Pin circle, radius = centre distance $a_w$", (g.a_w * np.cos(0.9), g.a_w * np.sin(0.9)), RED),
         ("Carrier plate outline (output, $n_c$)", (comp.carrier["plate_d"] / 2 * np.cos(-0.9), comp.carrier["plate_d"] / 2 * np.sin(-0.9)), GREEN)]
if comp.ring_mount["bolts"]:
    a = 2 * np.pi * 0.25 / comp.ring_mount["bolts"]
    right.insert(2, ("Ring clamping bolts", comp.ring_mount["bolt_circle_d"] / 2 * np.array([np.cos(a), np.sin(a)]), INK))
for (txt, xy, col), y in zip(right, (92, 76, 60, -64, -84)):
    label(ax, txt, xy, (L, y), color=col, ha="left")
dim(ax, (0, 0), (g.a_w * np.cos(-0.5), g.a_w * np.sin(-0.5)), "$a_w$", off=(4, 4), color=RED)
ax.text(0, -R_out - 14, f"Front view of the example stage: sun {g.sun.z}, planets {g.planet.z} (×{g.n_planets}), ring {g.ring.z}, module {g.sun.m:g} mm",
        ha="center", fontsize=9)
ax.set_xlim(-L - 80, L + 95); ax.set_ylim(-R_out - 20, R_out + 10)
ax.set_aspect("equal"); ax.axis("off")
plt.tight_layout()

The **axial section** shows how the parts stack along the shaft. The carrier is a cage: the input-side plate and the output-side plate hold both ends of every pin, and the posts (shown in the front view, lying between the planets) join the two plates. The face width $b$ is the axial length of the teeth.

In [ ]:
#| hide
fig, ax = plt.subplots(figsize=(9, 6.5))
t_c, b, w, cl = comp.carrier["plate_t"], g.sun.b, 1.0, 0.5
x0 = 0.0
x_p1 = x0 + cl
x_g = x_p1 + t_c + w
x_p2 = x_g + b + w
x_end = x_p2 + t_c + cl
Lst = x_end - x0
def box(x, y0, y1, length, fc, ec=INK, hatch=None, lw=0.7):
    for sgn in (1, -1):
        lo, hi = sorted((sgn * y0, sgn * y1))
        ax.add_patch(Rectangle((x, lo), length, hi - lo, fc=fc, ec=ec, lw=lw, hatch=hatch))
box(x0 - 18, 0, ci.shaft_d / 2, x_g + b + 2 - (x0 - 18), C_STEEL)                         # input shaft
box(x_g, ci.shaft_d / 2, g.sun.da / 2, b, C_SUN, hatch="////")                              # sun
box(x0, g.ring.df / 2, R_out, Lst, C_RING, hatch="\\\\\\\\")                                   # ring body
for y0, y1 in ((g.sun.da / 2 + 1.5, comp.carrier["plate_d"] / 2),):
    box(x_p1, y0, y1, t_c, C_CARRIER, ec=GREEN, hatch="xx")                                   # input-side plate
    box(x_p2, ci.shaft_d / 2 + 2, y1, t_c, C_CARRIER, ec=GREEN, hatch="xx")                  # output plate
box(x_p2 + t_c, 0, 0.6 * comp.carrier["output_shaft_min_d"] + 4, 22, C_CARRIER, ec=GREEN)     # output shaft
for sgn in (1, -1):
    yc = sgn * g.a_w
    ax.add_patch(Rectangle((x_g, yc - g.planet.da / 2), b, g.planet.da, fc=C_PLANET, ec=INK, lw=0.7, hatch="////"))
    ax.add_patch(Rectangle((x_g + 0.5, yc - comp.bearing["Ew"] / 2), comp.bearing["Bc"], comp.bearing["Ew"], fc="#999", ec=INK, lw=0.6))
    ax.add_patch(Rectangle((x_p1, yc - comp.pin["d"] / 2), comp.pin["length"], comp.pin["d"], fc=C_STEEL, ec=INK, lw=0.7))
    for xw in (x_g - w, x_g + b):
        ax.add_patch(Rectangle((xw, yc - comp.bearing["Ew"] / 2 - 2), w, comp.bearing["Ew"] + 4, fc="#555", ec=INK, lw=0.4))
box(x_g, g.ring.da / 2, g.ring.df / 2, b, C_RING, hatch="\\\\\\\\")                            # ring teeth (on top)
ax.plot([x0 - 22, x_end + 26], [0, 0], color=RED, lw=0.5, ls="-.")
for sgn in (1, -1):
    ax.plot([x0 - 4, x_end + 4], [sgn * g.a_w] * 2, color=RED, lw=0.4, ls="-.")
# labels
Lx = x_end + 34
label(ax, "Ring gear (fixed)", (x0 + 3, R_out - 3), (Lx, R_out - 2))
label(ax, "Planet gear", (x_g + b / 2, g.a_w + g.planet.da / 2 - 3), (Lx, 64))
label(ax, "Planet pin", (x_p1 + comp.pin["length"] - 2, g.a_w), (Lx, 50))
label(ax, "Needle roller bearing", (x_g + 3, g.a_w + comp.bearing["Ew"] / 2 - 1), (Lx, 38))
label(ax, "Thrust washers", (x_g + b + 0.5, g.a_w - comp.bearing["Ew"] / 2 - 1.5), (Lx, 22))
label(ax, "Carrier plate, input side", (x_p1 + t_c / 2, g.sun.da / 2 + 6), (x0 - 25, 40), ha="right", color=GREEN)
label(ax, "Carrier plate, output side", (x_p2 + t_c / 2, -g.sun.da / 2 - 4), (Lx, -22), color=GREEN)
label(ax, "Output shaft (carrier)", (x_p2 + t_c + 15, -8), (Lx, -10), color=GREEN)
label(ax, "Sun gear", (x_g + b / 2, g.sun.da / 2 - 3), (x0 - 25, 22), ha="right")
label(ax, "Input shaft (motor)", (x0 - 10, 3), (x0 - 25, 8), ha="right")
label(ax, "Ring teeth", (x_g + b / 2, -g.ring.da / 2 - 2), (Lx, -64))
dim(ax, (x_g, -R_out - 5), (x_g + b, -R_out - 5), "", color=BLUE)
ax.text(x_g + b / 2, -R_out - 9, "face width $b$", color=BLUE, fontsize=9, ha="center", va="top")
dim(ax, (x0, -R_out - 17), (x_end, -R_out - 17), "", color=BLUE)
ax.text(x_end + 3, -R_out - 17, "axial stack $L$", color=BLUE, fontsize=9, ha="left", va="center")
dim(ax, (x_p1, R_out + 5), (x_p1 + t_c, R_out + 5), "$t_c$", off=(0, 4))
dim(ax, (x0 - 6, 0), (x0 - 6, -g.a_w), "$a_w$", off=(-5, 0), color=RED)
ax.set_xlim(x0 - 75, Lx + 55); ax.set_ylim(-R_out - 26, R_out + 12)
ax.set_aspect("equal"); ax.axis("off")
ax.set_title("Axial section through two planets", fontsize=10)
plt.tight_layout()

## 2. Torque Flow, Speeds and Arrangements

With the ring fixed, the sun turns $i$ times for each turn of the carrier:

$$i = \frac{n_s}{n_c} = 1 + \frac{z_r}{z_s}.$$

The input torque $T_s$ enters at the sun and splits over the $N$ planets. Each planet pushes on its pin with the sum of the sun and ring tooth forces, which turns the carrier. The ring takes the difference as a reaction, so (ignoring losses)

$$T_c = i\,T_s,\qquad T_r = -(T_c - T_s) = -(i-1)\,T_s .$$

The planets spin on their pins at $n_{p/c} = -(n_s - n_c)\,z_s/z_p$, in the opposite direction to the sun.

The same three members give other ratios when a different one is held:

| Held | Input → output | Ratio | Range | Remarks |
|---|---|---|---|---|
| **Ring** | sun → carrier | $1 + z_r/z_s$ | about 3 to 10 | the usual reducer; used in this module |
| Carrier | sun → ring | $-z_r/z_s$ | about −2 to −9 | a reversing "star" drive: planets do not orbit, so no carrier centrifugal load |
| Sun | ring → carrier | $1 + z_s/z_r$ | about 1.1 to 1.7 | small reduction, for overdrive set-ups the other way round |

Above about $i = 6.8$ only three planets fit, and the ring grows to roughly $(i-1)$ times the sun diameter, so two stages or a stepped-planet stage are better for higher ratios.

## 3. Tooth Nomenclature

Every gear in the set is a spur gear with an *involute* tooth profile, the curve traced by the end of a taut string unwound from a circle (the **base circle**). The involute is used because two such flanks touch along a straight line (the line of action) at a constant angle, so the gears transmit motion at a constant speed ratio even if the centre distance is slightly off.

**Module $m$** is the size of the teeth: the reference diameter divided by the number of teeth, $m = d/z$, or equivalently the pitch divided by π, $m = p/\pi$. Two gears can only mesh if they have the same module and pressure angle. Module 1 mm gives a tooth about 2.25 mm deep and a pitch of 3.14 mm. Standard values come from ISO 54.

**Pressure angle $\alpha$** (20° in ISO 53) is the angle between the flank and the radius at the reference circle; equivalently, the angle between the line along which the teeth push and the tangent to the reference circle. A larger angle gives a stronger, thicker-rooted tooth but higher bearing loads and a lower contact ratio.

In [ ]:
#| hide
def tooth_figure():
    ge = GearGeometry(z=20, m=10.0, x=0.0, b=60, name="external")
    up = np.pi / 2
    pitch = 2 * np.pi / ge.z
    fig, ax = plt.subplots(figsize=(13, 4.4))
    P = rot(gear_outline(ge, n_flank=60, n_arc=20), up)
    r, ra, rf, rb = ge.d / 2, ge.da / 2, ge.df / 2, ge.db / 2
    th0, th1 = 62, 118
    body = Polygon(P, closed=True, fc=C_PLANET, ec=INK, lw=1.0)
    ax.add_patch(body)
    body.set_clip_path(Wedge((0, 0), ra + 4, th0 - 6, th1 + 6, width=ra + 4 - (rf - 9), transform=ax.transData))
    for rr, col, ls in ((ra, BLUE, "--"), (r, RED, "-."), (rb, GREEN, "--"), (rf, BLUE, "--")):
        ax.add_patch(Arc((0, 0), 2 * rr, 2 * rr, theta1=th0, theta2=th1, ls=ls, color=col, lw=0.9))
    # circle names at the right-hand end of each arc
    for rr, col, nm, dy in ((ra, BLUE, "tip circle, diameter $d_a$", 1.5), (r, RED, "reference circle, diameter $d = z\\,m$", 0),
                            (rb, GREEN, "base circle, diameter $d_b = d\\cos\\alpha$", -1.0), (rf, BLUE, "root circle, diameter $d_f$", -2.5)):
        a = np.radians(th0)
        ax.text(rr * np.cos(a) + 2, rr * np.sin(a) + dy, nm, color=col, fontsize=9, ha="left", va="center")
    ax.plot([0, 0], [rf - 6, ra + 14], color=RED, lw=0.5, ls="-.")
    psi = ge.s / ge.d
    def arcdim(a0, a1, rr, text, col, toff):
        ax.add_patch(Arc((0, 0), 2 * rr, 2 * rr, theta1=np.degrees(a0), theta2=np.degrees(a1), color=col, lw=1.6))
        for a in (a0, a1):
            ax.plot([(rr - 2.5) * np.cos(a), (rr + 2.5) * np.cos(a)], [(rr - 2.5) * np.sin(a), (rr + 2.5) * np.sin(a)], color=col, lw=0.9)
        am = (a0 + a1) / 2
        ax.text((rr + toff) * np.cos(am), (rr + toff) * np.sin(am), text, color=col, fontsize=9, ha="center", va="center",
                bbox=dict(fc="white", ec="none", pad=0.5, alpha=0.9))
    arcdim(up - psi, up + psi, r, "tooth\nthickness $s$", RED, 4.5)
    arcdim(up + psi, up + pitch - psi, r, "space\nwidth $e$", BLUE, -4.5)
    arcdim(up - psi - pitch, up - psi, ra + 9, "pitch $p = \\pi m$", INK, 3.5)
    # radial dimensions in the space to the left of the centre tooth's neighbour
    a_d = up + 1.5 * pitch
    u = np.array([np.cos(a_d), np.sin(a_d)])
    dim(ax, r * u, ra * u, "", color=BLUE)
    dim(ax, rf * u, r * u, "", color=BLUE)
    a_h = np.radians(th1 + 3.5)
    uh = np.array([np.cos(a_h), np.sin(a_h)])
    dim(ax, rf * uh, ra * uh, "", color=BLUE)
    # pressure angle on the right flank of the centre tooth, at the reference circle
    th = up - psi
    Pp = r * np.array([np.cos(th), np.sin(th)])
    T = rb * np.array([np.cos(th + ge.alpha), np.sin(th + ge.alpha)])
    v = Pp - T
    ax.plot([T[0], Pp[0] + 0.9 * v[0]], [T[1], Pp[1] + 0.9 * v[1]], color=GREEN, lw=1.1)
    ax.plot(*T, "o", color=GREEN, ms=3)
    tdir = np.array([np.sin(th), -np.cos(th)])
    ax.plot([Pp[0] - 10 * tdir[0], Pp[0] + 26 * tdir[0]], [Pp[1] - 10 * tdir[1], Pp[1] + 26 * tdir[1]], color=INK, lw=0.6, ls="--")
    a_t, a_n = np.degrees(np.arctan2(tdir[1], tdir[0])), np.degrees(np.arctan2(v[1], v[0]))
    ax.add_patch(Arc(Pp, 22, 22, theta1=min(a_t, a_n), theta2=max(a_t, a_n), color=GREEN, lw=1.3))
    am = np.radians((a_t + a_n) / 2)
    ax.text(Pp[0] + 15 * np.cos(am), Pp[1] + 15 * np.sin(am), "$\\alpha$", color=GREEN, fontsize=13, ha="center", va="center")
    # leaders on the right, top to bottom
    ax_r = 62
    label(ax, "line of action: normal to the flank,\ntangent to the base circle (at T)", Pp + 0.75 * v, (ax_r, 122), color=GREEN)
    label(ax, "tangent to the reference circle", Pp + 22 * tdir, (ax_r, 113), color=INK)
    label(ax, "pressure angle $\\alpha$ = 20° between them", Pp + 9 * np.array([np.cos(am), np.sin(am)]), (ax_r, 106), color=GREEN)
    # leaders on the left, top to bottom
    rr_f = ra - 4
    hf_ = psi + float(inv(ge.alpha)) - float(inv(np.arccos(rb / rr_f)))
    flank_pt = rr_f * np.array([np.cos(up + hf_), np.sin(up + hf_)])
    fil = (rf + 1.2) * np.array([np.cos(up + pitch / 2 - 0.035), np.sin(up + pitch / 2 - 0.035)])
    ax_l = -66
    label(ax, "tooth tip", (0, ra), (ax_l, 124), ha="right")
    label(ax, "flank (involute)", flank_pt, (ax_l, 116), ha="right")
    label(ax, "addendum $h_a = m$", (r + ra) / 2 * u, (ax_l, 106), ha="right", color=BLUE)
    label(ax, "dedendum $h_f = 1.25\\,m$", (r + rf) / 2 * u, (ax_l, 97), ha="right", color=BLUE)
    label(ax, "tooth depth $h = h_a + h_f = 2.25\\,m$", (ra + rf) / 2 * uh, (ax_l, 88), ha="right", color=BLUE)
    label(ax, "root fillet (left by the hob tip radius)", fil, (ax_l, 79), ha="right")
    ax.text(0.5, -0.01, f"External spur gear: z = {ge.z}, m = {ge.m:g} mm (drawn large), α = 20°, x = 0. "
            "Face width b is the tooth length into the page (see the axial section).", ha="center", va="top", fontsize=9, transform=ax.transAxes)
    ax.set_xlim(-150, 165); ax.set_ylim(rf - 17, ra + 18)
    ax.set_aspect("equal"); ax.axis("off")
    plt.tight_layout()
tooth_figure()

| Term | Symbol | Definition (external gear) |
|---|---|---|
| Number of teeth | $z$ | |
| Module | $m$ | $d/z$ |
| Pitch | $p$ | arc from one tooth to the next on the reference circle, $\pi m$ |
| Base pitch | $p_b$ | the same along the base circle and along the line of action, $p\cos\alpha$ |
| Reference diameter | $d$ | $z\,m$; where the module and pressure angle are defined |
| Base diameter | $d_b$ | $d\cos\alpha$; the circle the involute unwinds from — no involute inside it |
| Tip diameter | $d_a$ | outside diameter of the gear, $d + 2m(h_{aP}^* + x - k^*)$ |
| Root diameter | $d_f$ | diameter at the bottom of the tooth spaces, $d - 2m(h_{fP}^* - x)$ |
| Addendum | $h_a$ | tip to reference circle, $(d_a - d)/2$; standard $1.0\,m$ |
| Dedendum | $h_f$ | reference circle to root, $(d - d_f)/2$; standard $1.25\,m$ (0.25 m is the clearance for the mating tip) |
| Tooth depth | $h$ | $h_a + h_f$; standard $2.25\,m$ |
| Tooth thickness | $s$ | arc length of one tooth on the reference circle, $m(\pi/2 + 2x\tan\alpha)$; made slightly thinner to give backlash |
| Space width | $e$ | arc length of the gap on the reference circle, $p - s$ |
| Root fillet radius | $\rho_F$ | radius joining the flank to the root; left by the tool tip radius $\rho_{fP} = 0.38\,m$ |
| Root form diameter | $d_{Ff}$ | where the involute ends and the fillet begins |
| Face width | $b$ | axial length of the teeth (see the axial section) |
| Pressure angle | $\alpha$ | angle between flank and radius at the reference circle, 20° |

### Profile shift

The tool that cuts the gear (a hob, which acts like a straight-sided rack) can be moved away from or toward the gear centre by $x\,m$. This **profile shift** $x$ uses a different part of the same involute: a positive shift makes the tooth thicker at the root and thinner at the tip and pushes the tip and root circles outward, without changing $d$, $d_b$ or $m$. It is used to stop the cutter from undercutting a small gear, to balance strength between sun and planet, and to make both meshes of a planetary set fit one centre distance.

In [ ]:
#| hide
fig, axs = plt.subplots(1, 3, figsize=(11, 4))
for ax, x in zip(axs, (-0.3, 0.0, 0.5)):
    gx = GearGeometry(z=14, m=10.0, x=x)
    P = rot(gear_outline(gx, n_flank=60), np.pi / 2)
    ax.add_patch(Polygon(P, closed=True, fc=C_SUN, ec=INK, lw=0.9))
    for rr, col in ((gx.d / 2, RED), (gx.db / 2, GREEN), (gx.da / 2, BLUE), (gx.df / 2, BLUE)):
        ax.add_patch(Arc((0, 0), 2 * rr, 2 * rr, theta1=50, theta2=130, color=col, lw=0.8, ls="--" if col != RED else "-."))
    ax.set_xlim(-38, 38); ax.set_ylim(38, 92); ax.set_aspect("equal"); ax.axis("off")
    ttl = f"x = {x:+.1f}" + ("  (undercut: z < z_min)" if gx.undercut else "")
    ax.set_title(ttl + f"\ns = {gx.s:.1f}, s_a = {gx.sa:.1f}, d_a = {gx.da:.0f}", fontsize=9)
fig.suptitle("Same 14-tooth gear, module 10 (scaled), cut with three profile shifts: red = reference circle d (unchanged), "
             "green = base circle, blue = tip and root circles", fontsize=9)
plt.tight_layout()

### The internal (ring) gear

On the ring the teeth point inward and the tooth spaces are cut into the ring. Everything is the same as on an external gear, but mirrored: the **tip circle $d_a$ is the smaller, inner circle** and the **root circle $d_f$ the larger, outer one**. The rim between the root circle and the outside of the ring carries the reaction torque and is checked with the ISO 6336 rim factor. Ring tips are often shortened (tip reduction $k^*$) so they clear the planet's root fillet.

In [ ]:
#| hide
fig, ax = plt.subplots(figsize=(9, 5.5))
gi = GearGeometry(z=48, m=10.0, internal=True)
P = rot(gear_outline(gi, n_flank=50, n_arc=16), np.pi / 2 + np.pi / gi.z * 0)
R_o = gi.df / 2 + 3.5 * gi.m
ax.add_patch(Circle((0, 0), R_o, fc=C_RING, ec=INK, lw=1.0))
ax.add_patch(Polygon(P, closed=True, fc="white", ec=INK, lw=1.0))
for rr, col, nm in ((gi.da / 2, BLUE, "tip circle $d_a$ (inner)"), (gi.d / 2, RED, "reference circle $d$"),
                    (gi.df / 2, BLUE, "root circle $d_f$ (outer)")):
    ax.add_patch(Arc((0, 0), 2 * rr, 2 * rr, theta1=73, theta2=107, color=col, lw=0.9, ls="--" if col != RED else "-."))
    ax.text(rr * np.cos(np.radians(72.5)), rr * np.sin(np.radians(72.5)), nm, color=col, fontsize=9, va="center")
psi = gi.s / gi.d
for a0, a1, nm, col, off in ((np.pi / 2 - psi, np.pi / 2 + psi, "tooth thickness $s$", RED, -10),
                             (np.pi / 2 + psi, np.pi / 2 + 2 * np.pi / gi.z - psi, "space width $e$", BLUE, -10)):
    ax.add_patch(Arc((0, 0), gi.d, gi.d, theta1=np.degrees(a0), theta2=np.degrees(a1), color=col, lw=1.4))
    am = (a0 + a1) / 2
    ax.text((gi.d / 2 + off) * np.cos(am), (gi.d / 2 + off) * np.sin(am), nm, color=col, fontsize=9, ha="center",
            bbox=dict(fc="white", ec="none", pad=0.5, alpha=0.85))
a_r = np.pi / 2 - 2.6 * np.pi / gi.z
u = np.array([np.cos(a_r), np.sin(a_r)])
dim(ax, gi.df / 2 * u, R_o * u, "rim $s_R$", off=(16, 0))
a_h = np.pi / 2 - 1.5 * np.pi / gi.z
u = np.array([np.cos(a_h), np.sin(a_h)])
dim(ax, gi.da / 2 * u, gi.df / 2 * u, "depth $h$", off=(15, 0))
label(ax, "ring tooth (material)", (0, gi.d / 2 + 6), (-150, 250), ha="right")
label(ax, "tooth space (cut)", gi.d / 2 * np.array([np.cos(np.pi / 2 + np.pi / gi.z), np.sin(np.pi / 2 + np.pi / gi.z)]), (-150, 215), ha="right")
label(ax, "ring body", (0, R_o - 8), (-150, 265), ha="right")
ax.set_xlim(-230, 160); ax.set_ylim(205, 275); ax.set_aspect("equal"); ax.axis("off")
ax.set_title(f"Internal gear, z = {gi.z}, m = {gi.m:g} (scaled up)", fontsize=10)
plt.tight_layout()

## 4. Two Gears in Mesh

When two gears mesh, their reference circles (with no profile shift) roll on each other without slipping at the **pitch point** $C$ on the line joining the centres. The teeth push on each other along the **line of action**, the common tangent of the two base circles through $C$, which makes the **operating pressure angle $\alpha_w$** with the common tangent of the pitch circles. Contact starts where the tip circle of the driven gear crosses the line of action ($A$) and ends where the tip circle of the driver crosses it ($E$). The length $AE$ is the **path of contact $g_\alpha$**; divided by the base pitch it is the **contact ratio** $\varepsilon_\alpha = g_\alpha/p_b$, the average number of tooth pairs in contact (keep it above 1.2).

The **centre distance** $a_w$ is the distance between the gear axes. With profile shift it differs from the reference value $a = (z_1 + z_2)\,m/2$, and $\alpha_w$ differs from $\alpha$ accordingly. **Backlash** $j$ is the play between the non-driving flanks; it comes from making each tooth slightly thinner than $s$ (the thickness allowances on the gear drawing).

In [ ]:
#| hide
fig, ax = plt.subplots(figsize=(12, 6.2))
g1, g2 = GearGeometry(z=14, m=10, x=0.3, name="pinion"), GearGeometry(z=22, m=10, x=-0.1, name="wheel")
mc = mesh_check(g1, g2)
aw, alw = mc.a_w, mc.alpha_w
c1, c2 = np.array([0.0, 0.0]), np.array([0.0, aw])
ds = -0.8                                     # visible backlash
P1 = rot(gear_outline(g1, ds, n_flank=50), np.pi / 2)
# wheel: a tooth space faces the pinion tooth at the pitch point
P2 = rot(gear_outline(g2, ds, n_flank=50), -np.pi / 2 + np.pi / g2.z) + c2
ax.add_patch(Polygon(P1, closed=True, fc=C_SUN, ec=INK, lw=0.8))
ax.add_patch(Polygon(P2, closed=True, fc=C_PLANET, ec=INK, lw=0.8))
rw1 = aw * g1.z / (g1.z + g2.z)
C = np.array([0, rw1])
for c, gg in ((c1, g1), (c2, g2)):
    circ(ax, gg.db / 2, "--", GREEN, 0.8, c)
    circ(ax, gg.da / 2, ":", BLUE, 0.9, c)
circ(ax, rw1, "-.", RED, 0.8, c1)
circ(ax, aw - rw1, "-.", RED, 0.8, c2)
T1 = c1 + g1.db / 2 * np.array([np.sin(alw), np.cos(alw)])
T2 = c2 - g2.db / 2 * np.array([np.sin(alw), np.cos(alw)])
ax.plot([T1[0], T2[0]], [T1[1], T2[1]], color=GREEN, lw=1.3)
for T, nm, cc in ((T1, "$T_1$", c1), (T2, "$T_2$", c2)):
    ax.plot(*T, "o", color=GREEN, ms=3.5); ax.text(T[0] + 2.5, T[1] + (-7 if T is T1 else 1), nm, color=GREEN, fontsize=10)
def hit(c, ra):
    u = (T2 - T1) / np.linalg.norm(T2 - T1)
    f = T1 - c
    bq = 2 * f @ u; cq = f @ f - ra ** 2
    ts = [(-bq + s_ * math.sqrt(bq * bq - 4 * cq)) / 2 for s_ in (1, -1)]
    return min((T1 + t_ * u for t_ in ts), key=lambda p_: np.linalg.norm(p_ - C))
A, E = hit(c2, g2.da / 2), hit(c1, g1.da / 2)
ax.plot([A[0], E[0]], [A[1], E[1]], color=RED, lw=4, alpha=0.55, solid_capstyle="butt")
for p_, nm, o in ((A, "A", (-7, -6)), (E, "E", (2, 3)), (C, "C", (-3, 4))):
    ax.plot(*p_, "o", color=RED, ms=4); ax.text(p_[0] + o[0], p_[1] + o[1], nm, color=RED, fontsize=11)
ax.plot([-60, 75], [rw1, rw1], color=INK, lw=0.5, ls="--")
ax.add_patch(Arc(C, 84, 84, theta1=-np.degrees(alw), theta2=0, color=GREEN, lw=1.3))
ax.text(C[0] + 44, C[1] - 9, "$\\alpha_w$", color=GREEN, fontsize=13)
ax.plot([0, 0], [rw1 - 75, rw1 + 75], color=RED, lw=0.4, ls="-.")
# leaders: right side, top to bottom
xr = 110
label(ax, "line of action (common tangent\nof the two base circles)", T2 + 0.2 * (T1 - T2), (xr, rw1 + 48), color=GREEN)
label(ax, "operating pressure angle $\\alpha_w$", C + [41.5, -6], (xr, rw1 + 30), color=GREEN)
label(ax, "path of contact $g_\\alpha$ = AE;\ncontact ratio $\\varepsilon_\\alpha = g_\\alpha / p_b$", (A + E) / 2 + (A - E) * 0.25, (xr, rw1 + 8), color=RED)
label(ax, "tangent to the pitch circles at C", (60, rw1), (xr, rw1 - 14), color=INK)
label(ax, "line of centres (wheel centre $O_2$ above,\npinion centre $O_1$ below)", (0, rw1 - 60), (xr, rw1 - 40), color=RED)
# leaders: left side, top to bottom
xl = -95
label(ax, "wheel (driven)", c2 + (g2.d / 2 - 12) * np.array([np.cos(-2.0), np.sin(-2.0)]), (xl, rw1 + 64), ha="right")
label(ax, "tip circles", c2 + (g2.da / 2) * np.array([np.cos(-2.15), np.sin(-2.15)]), (xl, rw1 + 50), ha="right", color=BLUE)
label(ax, "pitch point C: the pitch circles\nroll on each other here", C + [-1, 1], (xl, rw1 + 30), ha="right", color=RED)
label(ax, "backlash $j$: play between the\nnon-driving flanks", C + [-9.5, 2.5], (xl, rw1 + 10), ha="right")
label(ax, "pitch circles (radii $r_{w1}, r_{w2}$)", (rw1 * np.cos(2.0), rw1 * np.sin(2.0)), (xl, rw1 - 22), ha="right", color=RED)
label(ax, "base circles", (g1.db / 2 * np.cos(2.2), g1.db / 2 * np.sin(2.2)), (xl, rw1 - 38), ha="right", color=GREEN)
label(ax, "pinion (driver)", ((g1.d / 2 - 14) * np.cos(2.0), (g1.d / 2 - 14) * np.sin(2.0)), (xl, rw1 - 56), ha="right")
ax.text(0.5, -0.01, f"Pinion z = {g1.z}, x = {g1.x:+.1f}; wheel z = {g2.z}, x = {g2.x:+.1f}; m = {g1.m:g} mm (drawn large). "
        f"Centre distance $a_w$ = {aw:.2f} mm (reference a = {(g1.z + g2.z) * g1.m / 2:.0f} mm), "
        f"$\\alpha_w$ = {math.degrees(alw):.2f}°, $\\varepsilon_\\alpha$ = {mc.eps_alpha:.2f}", ha="center", va="top", fontsize=9, transform=ax.transAxes)
ax.set_xlim(-245, 260); ax.set_ylim(rw1 - 80, rw1 + 75); ax.set_aspect("equal"); ax.axis("off")
plt.tight_layout()

In a planetary stage the planet meshes with the sun (external pair, as drawn) and with the ring (internal pair). For the internal pair the two base circles lie on the same side of the line of action, both gears turn the same way, and the contact ratio is higher. Both meshes must share one centre distance $a_w$, the radius of the pin circle on the carrier.

## 5. Data That Defines a Design

A single-stage spur planetary (helix angle β = 0) is fully defined by the data below. Group A is what you supply (`PlanetaryRequirements`); group B is what the design search chooses; groups C to G are what `write_package` writes out for manufacture.

**A. Requirements**

1. Arrangement: fixed member, input, output (default ring fixed, sun in, carrier out)
2. Target ratio $i$ and allowed tolerance
3. Output torque: nominal, peak, duty cycle or load spectrum
4. Input speed: nominal and maximum
5. Required life [h] and application factor $K_A$
6. Envelope: maximum outer diameter and axial length
7. Input shaft: diameter and joint (key, spline, or sun cut on the shaft)
8. Output interface: shaft or flange, bearing arrangement
9. Environment and lubrication: grease or oil, temperature, sealing
10. Limits on backlash, noise or efficiency, if any

**B. Gear set**

11. Number of planets $N$
12. Tooth counts $z_s, z_p, z_r$
13. Module $m$ (common)
14. Pressure angle $\alpha$ = 20° and basic rack (ISO 53 profile A)
15. Profile shift coefficients $x_s, x_p, x_r$
16. Tip reduction $k^*$ (usually on the ring)
17. Face width $b$ of each gear
18. Operating centre distance $a_w$ (pin circle radius)

**C. Per gear, for its drawing**

19. Diameters $d, d_b, d_a, d_f$
20. Root form diameter $d_{Ff}$ and start of active profile $d_{Nf}$
21. Tooth thickness $s$ with allowances $A_{sne}/A_{sni}$
22. Inspection: span $W_k$ over $k$ teeth and/or dimension over (between) balls $M_d$ with ball size $D_M$
23. ISO 1328-1 flank tolerance class with $f_{pT}, F_{pT}, F_{\alpha T}, F_{\beta T}, F_{rT}$
24. Tip diameter and face width tolerances, tip chamfer
25. Material, heat treatment, surface and core hardness, case depth
26. Flank and root roughness, process route

**D. Mesh checks**

27. Contact ratio $\varepsilon_\alpha$ ≥ 1.2 in both meshes
28. Tip clearance, no tip or root interference
29. Assembly condition and neighbour clearance
30. Backlash range of both meshes

**E. Strength (ISO 6336)**

31. Load factors $K_A, K_v, K_{H\beta}, K_{H\alpha}, K_\gamma$
32. $S_H$ and $S_F$ of every gear in every mesh against the required minima

**F. Supporting parts**

33. Planet bearing: type, $F_w \times E_w \times B_c$, $C_r$ against $C_{req}$, $L_{10h}$
34. Planet pin: diameter, length, material, hardness, fits (pin j5, plate holes N7)
35. Planet bore fit (G6), thrust washers
36. Carrier: plate diameter and thickness, pin hole positions and tolerance, posts, output shaft
37. Sun–shaft joint: key size and length or spline data, bore fit
38. Ring fixing: bolts (size, number, bolt circle, torque) or press fit and dowels; ring OD and rim
39. Axial stack against the envelope length

**G. Package**

40. Gear data tables, part dimensions and fits, parts list, layout drawing

## 6. Parameter Glossary

| Parameter | Symbol | Units | Meaning | Calculated in |
|---|---|---|---|---|
| Ratio | $i$ | – | sun speed ÷ carrier speed, $1 + z_r/z_s$ | `02_layout` |
| Number of planets | $N$ | – | planets equally spaced on the carrier | `02_layout` |
| Tooth counts | $z_s, z_p, z_r$ | – | sun, planet, ring | `02_layout` |
| Module | $m$ | mm | tooth size, $d/z$ | `02_layout`, `06_design` |
| Pressure angle | $\alpha$ | ° | 20° (ISO 53) | `01_involute` |
| Operating pressure angle | $\alpha_w$ | ° | pressure angle at the actual centre distance | `01_involute` |
| Profile shift coefficient | $x$ | – | tool offset ÷ module | `02_layout` |
| Tip reduction | $k^*$ | – | tip shortening ÷ module | `02_layout` |
| Reference / base / tip / root diameter | $d, d_b, d_a, d_f$ | mm | see tooth nomenclature | `01_involute` |
| Root form diameter / start of active profile | $d_{Ff}, d_{Nf}$ | mm | end of the involute / first contact point | `01_involute` |
| Face width | $b$ | mm | axial tooth length | `06_design` |
| Tooth thickness | $s$ | mm | arc on the reference circle | `01_involute` |
| Thickness allowances | $A_{sne}, A_{sni}$ | mm | how much thinner than $s$ the tooth is made (upper, lower) | `05_manufacturing` |
| Span measurement | $W_k$ | mm | across $k$ teeth with a disc micrometer | `01_involute`, `05_manufacturing` |
| Dimension over / between balls | $M_d$ | mm | across two balls of diameter $D_M$ in opposite spaces | `01_involute`, `05_manufacturing` |
| Centre distance | $a_w$ | mm | pin circle radius | `01_involute`, `02_layout` |
| Contact ratio | $\varepsilon_\alpha$ | – | tooth pairs in contact on average | `01_involute` |
| Backlash | $j_{bn}$ | mm | normal play between flanks | `05_manufacturing` |
| Tangential force | $F_t$ | N | tooth force per planet mesh on the reference circle | `03_rating` |
| Contact stress / safety | $\sigma_H$, $S_H$ | MPa / – | flank pitting | `03_rating` |
| Root stress / safety | $\sigma_F$, $S_F$ | MPa / – | tooth breakage | `03_rating` |
| Rim thickness | $s_R$ | mm | material under the tooth roots | `04_components` |
| Flank tolerance class | $A$ | – | ISO 1328-1 accuracy (lower = more accurate) | `05_manufacturing` |
| Planet bearing | $F_w\times E_w\times B_c$ | mm | pin Ø × bore Ø × width of the needle cage | `04_components` |
| Carrier plate | $D_c, t_c$ | mm | diameter, thickness | `04_components` |

---

## Tests

In [ ]:
#| hide
# the example stage used for the drawings is valid, and its parts fit
assert ps.ok and comp.carrier["plate_d"] <= ps.ring.da - 2
# nomenclature relations quoted in the tables
g0 = GearGeometry(z=20, m=10.0)
assert math.isclose(g0.d, 200) and math.isclose(g0.p, math.pi * 10) and math.isclose(g0.h, 22.5)
assert math.isclose(g0.da - g0.d, 2 * 10) and math.isclose(g0.d - g0.df, 2 * 12.5)
assert math.isclose(g0.s + (g0.p - g0.s), g0.p)
gi0 = GearGeometry(z=48, m=10.0, internal=True)
assert gi0.da < gi0.d < gi0.df
# profile shift does not change d or d_b, and thickens the tooth
assert math.isclose(GearGeometry(z=14, m=10, x=0.5).d, GearGeometry(z=14, m=10).d)
assert GearGeometry(z=14, m=10, x=0.5).s > GearGeometry(z=14, m=10).s
# torque relations of section 2
i = ps.ratio
assert math.isclose(rat.T_carrier, i * rat.T_sun * 0.97, rel_tol=1e-9)